In [1]:
!pip install rdkit torch tqdm pandas numpy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, QED
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from tqdm import tqdm
import math
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 18.4 MB/s eta 0:00:00
Using device: cpu


In [3]:
import kagglehub
path = kagglehub.dataset_download("yanmaksi/big-molecules-smiles-dataset")

100%|██████████| 289k/289k [00:00<00:00, 66.7MB/s]

Extracting files...


In [9]:
import os

if not os.path.exists("SMILES_Big_Data_Set.csv"):
    # Running on Google Colab — upload the file
    from google.colab import files
    print("Upload SMILES_Big_Data_Set.csv from your computer...")
    files.upload()   # file picker will appear — select SMILES_Big_Data_Set.csv

df = pd.read_csv("SMILES_Big_Data_Set.csv")
smiles_list = df["SMILES"].dropna().unique().tolist()
print(f"Total SMILES loaded: {len(smiles_list)}")

morgan_gen = GetMorganGenerator(radius=2, fpSize=2048)

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        fp = morgan_gen.GetFingerprint(mol)
        return np.array(fp, dtype=np.float32)
    return None

print("Converting SMILES to fingerprints...")
fingerprints, valid_smiles = [], []
for s in tqdm(smiles_list):
    fp = smiles_to_fp(s)
    if fp is not None:
        fingerprints.append(fp)
        valid_smiles.append(s)

fingerprints = np.array(fingerprints)
print(f"Valid fingerprints: {fingerprints.shape}")

Upload SMILES_Big_Data_Set.csv from your computer...


Saving SMILES_Big_Data_Set.csv to SMILES_Big_Data_Set.csv
Total SMILES loaded: 15872
Converting SMILES to fingerprints...


100%|██████████| 15872/15872 [00:26<00:00, 610.14it/s]

Valid fingerprints: (15872, 2048)


In [10]:
timesteps = 1000
betas     = torch.linspace(0.0001, 0.02, timesteps).to(device)
alphas    = (1.0 - betas)
alphas_cumprod = torch.cumprod(alphas, dim=0)

def forward_diffusion(x0, t, noise=None):
    """Add noise to x0 at timestep t."""
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_ac   = alphas_cumprod[t].unsqueeze(1) ** 0.5
    sqrt_1mac = (1 - alphas_cumprod[t]).unsqueeze(1) ** 0.5
    return sqrt_ac * x0 + sqrt_1mac * noise, noise

In [11]:
class SinusoidalTimeEmbedding(nn.Module):
    """Gives the model a sense of which noise level it's working at."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / (half - 1)
        )
        args = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)

class DiffusionModel(nn.Module):
    def __init__(self, input_dim=2048, hidden_dim=1024, time_dim=128):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim * 2),
            nn.SiLU(),
            nn.Linear(time_dim * 2, time_dim),
        )
        self.net = nn.Sequential(
            nn.Linear(input_dim + time_dim, hidden_dim),
            nn.SiLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.1),

            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(0.1),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.SiLU(),
            nn.LayerNorm(hidden_dim // 2),

            nn.Linear(hidden_dim // 2, input_dim),
        )

    def forward(self, x, t):
        t_emb = self.time_embed(t)          # (B, time_dim)
        x_in  = torch.cat([x, t_emb], dim=-1)  # (B, input_dim + time_dim)
        return self.net(x_in)

model     = DiffusionModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
mse       = nn.MSELoss()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

Model parameters: 4,925,312


In [12]:
fp_tensor = torch.tensor(fingerprints, dtype=torch.float32)
dataset   = TensorDataset(fp_tensor)
loader    = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)

EPOCHS       = 100
best_loss    = float("inf")
patience     = 10   # early stopping
no_improve   = 0
loss_history = []

print(f"\nTraining for {EPOCHS} epochs on {len(fp_tensor)} molecules...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for (x0,) in loader:
        x0 = x0.to(device)
        t  = torch.randint(0, timesteps, (x0.shape[0],), device=device)

        noisy_x, noise = forward_diffusion(x0, t)
        noise_pred     = model(noisy_x, t)
        loss           = mse(noise_pred, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()

    scheduler.step()
    avg_loss = epoch_loss / len(loader)
    loss_history.append(avg_loss)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d}/{EPOCHS} | Loss: {avg_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    # Save best model
    if avg_loss < best_loss:
        best_loss  = avg_loss
        no_improve = 0
        torch.save({
            "model_state": model.state_dict(),
            "epoch":       epoch,
            "loss":        avg_loss,
            "config": {"input_dim": 2048, "hidden_dim": 1024, "time_dim": 128}
        }, "model_best.pth")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs)")
            break

# Also save final weights in the simple format app.py expects
checkpoint = torch.load("model_best.pth", map_location="cpu")
# Re-create model and load best weights for app.py compatibility
torch.save(checkpoint["model_state"], "model.pth")
print(f"\nBest model saved — epoch {checkpoint['epoch']}, loss {checkpoint['loss']:.6f}")


Training for 100 epochs on 15872 molecules...
Epoch  10/100 | Loss: 0.797866 | LR: 1.95e-04
Epoch  20/100 | Loss: 0.791337 | LR: 1.81e-04
Epoch  30/100 | Loss: 0.789043 | LR: 1.59e-04
Epoch  40/100 | Loss: 0.787415 | LR: 1.31e-04
Epoch  50/100 | Loss: 0.785937 | LR: 1.00e-04
Epoch  60/100 | Loss: 0.784966 | LR: 6.91e-05
Epoch  70/100 | Loss: 0.784701 | LR: 4.12e-05
Epoch  80/100 | Loss: 0.783935 | LR: 1.91e-05
Epoch  90/100 | Loss: 0.783432 | LR: 4.89e-06
Epoch 100/100 | Loss: 0.783668 | LR: 0.00e+00

Best model saved — epoch 91, loss 0.782770


In [13]:
print("\nEvaluating generated molecules...")

@torch.no_grad()
def sample_molecules(n=100):
    model.eval()
    x = torch.randn(n, 2048).to(device)
    for t in reversed(range(timesteps)):
        t_batch    = torch.full((n,), t, device=device, dtype=torch.long)
        pred_noise = model(x, t_batch)
        alpha      = alphas[t]
        alpha_hat  = alphas_cumprod[t]
        beta       = betas[t]
        noise      = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        x = (1/alpha**0.5) * (x - ((1-alpha)/(1-alpha_hat)**0.5) * pred_noise) + beta**0.5 * noise
    return x.cpu()

def fp_to_smiles(gen_fp, train_fps, train_smiles):
    """Find nearest training molecule by Tanimoto similarity."""
    bin_str = ''.join(['1' if b > 0.5 else '0' for b in gen_fp.numpy()])
    try:
        gen_rdkit = DataStructs.CreateFromBitString(bin_str)
    except:
        return None
    best_sim, best_smi = 0, None
    for rfp, rsmi in zip(train_fps, train_smiles):
        ref = DataStructs.CreateFromBitString(''.join(['1' if b else '0' for b in rfp]))
        sim = DataStructs.TanimotoSimilarity(gen_rdkit, ref)
        if sim > best_sim:
            best_sim, best_smi = sim, rsmi
    return best_smi

# Sample 200 molecules for evaluation
print("Sampling 200 molecules (this takes a few minutes)...")
gen_fps   = sample_molecules(200)
gen_smiles_raw = []
for fp in tqdm(gen_fps):
    smi = fp_to_smiles(fp, fingerprints[:2000], valid_smiles[:2000])
    if smi:
        gen_smiles_raw.append(smi)

# Metrics
train_set = set(valid_smiles)
valid_mols, qed_scores = [], []
unique_smiles = list(set(gen_smiles_raw))

for smi in unique_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        valid_mols.append(smi)
        qed_scores.append(QED.qed(mol))

validity  = len(valid_mols) / len(gen_smiles_raw) * 100 if gen_smiles_raw else 0
uniqueness = len(unique_smiles) / len(gen_smiles_raw) * 100 if gen_smiles_raw else 0
novelty   = sum(1 for s in valid_mols if s not in train_set) / len(valid_mols) * 100 if valid_mols else 0
avg_qed   = np.mean(qed_scores) if qed_scores else 0

print("\n" + "="*45)
print("       EVALUATION RESULTS")
print("="*45)
print(f"  Validity    : {validity:.1f}%   (chemically valid SMILES)")
print(f"  Uniqueness  : {uniqueness:.1f}%   (non-duplicate molecules)")
print(f"  Novelty     : {novelty:.1f}%   (not in training set)")
print(f"  Avg QED     : {avg_qed:.3f}   (drug-likeness, 0-1)")
print(f"  Best loss   : {best_loss:.6f}")
print("="*45)


Evaluating generated molecules...
Sampling 200 molecules (this takes a few minutes)...


100%|██████████| 200/200 [01:22<00:00,  2.43it/s]


       EVALUATION RESULTS
  Validity    : 27.0%   (chemically valid SMILES)
  Uniqueness  : 27.0%   (non-duplicate molecules)
  Novelty     : 0.0%   (not in training set)
  Avg QED     : 0.330   (drug-likeness, 0-1)
  Best loss   : 0.782770


In [14]:
print("""
IMPORTANT: Your app.py uses the OLD SimpleDiffusionModel architecture.
After training, update app.py to use the new DiffusionModel:

Replace the model class and loading code in app.py with:

    class SinusoidalTimeEmbedding(nn.Module):
        def __init__(self, dim):
            super().__init__()
            self.dim = dim
        def forward(self, t):
            half = self.dim // 2
            freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half-1))
            args = t[:,None].float() * freqs[None]
            return torch.cat([args.sin(), args.cos()], dim=-1)

    class DiffusionModel(nn.Module):
        def __init__(self, input_dim=2048, hidden_dim=1024, time_dim=128):
            super().__init__()
            self.time_embed = nn.Sequential(
                SinusoidalTimeEmbedding(time_dim),
                nn.Linear(time_dim, time_dim*2), nn.SiLU(),
                nn.Linear(time_dim*2, time_dim),
            )
            self.net = nn.Sequential(
                nn.Linear(input_dim+time_dim, hidden_dim), nn.SiLU(), nn.LayerNorm(hidden_dim), nn.Dropout(0.1),
                nn.Linear(hidden_dim, hidden_dim), nn.SiLU(), nn.LayerNorm(hidden_dim), nn.Dropout(0.1),
                nn.Linear(hidden_dim, hidden_dim//2), nn.SiLU(), nn.LayerNorm(hidden_dim//2),
                nn.Linear(hidden_dim//2, input_dim),
            )
        def forward(self, x, t):
            return self.net(torch.cat([x, self.time_embed(t)], dim=-1))

    model = DiffusionModel().to(device)
    model.load_state_dict(torch.load("model.pth", map_location=device))
    model.eval()
""")


IMPORTANT: Your app.py uses the OLD SimpleDiffusionModel architecture.
After training, update app.py to use the new DiffusionModel:
 
Replace the model class and loading code in app.py with:
 
    class SinusoidalTimeEmbedding(nn.Module):
        def __init__(self, dim):
            super().__init__()
            self.dim = dim
        def forward(self, t):
            half = self.dim // 2
            freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half-1))
            args = t[:,None].float() * freqs[None]
            return torch.cat([args.sin(), args.cos()], dim=-1)
 
    class DiffusionModel(nn.Module):
        def __init__(self, input_dim=2048, hidden_dim=1024, time_dim=128):
            super().__init__()
            self.time_embed = nn.Sequential(
                SinusoidalTimeEmbedding(time_dim),
                nn.Linear(time_dim, time_dim*2), nn.SiLU(),
                nn.Linear(time_dim*2, time_dim),
            )
            self.net = nn.Seque

# New Section